# CS 687 · Homework 1 — Project 1, Checkpoint 1

**Tokenization, embeddings, and the information-theoretic view**

This notebook follows the lecture notes section by section. You can run it in Google Colab, or on your own computer with Jupyter. Nothing here needs a graphics card.

The two tasks you have to write are marked clearly. Everything else is given to you. Each task is followed immediately by a cell that runs its relevant public tests, so you can correct that task before continuing. The final public-test cell runs the complete suite again; always run it before downloading and submitting your notebook.

| Notebook section | Notes section |
|---|---|
| 1. Setup | — |
| 2. Task 1: the byte-pair-encoding trainer | 2.2 |
| 3. Checking against the hand trace | 2.2 |
| 4. Comparing with GPT-2's tokenizer | 2.3 |
| 5. The fertility experiment | 2.3 |
| 6. Task 2: training pairs by sliding window | 3.2 |
| 7. Embeddings, and the shape check | 3.1, 3.2 |
| 8. Units: nats, perplexity, bits per byte | 1.3 |
| 9. The embedding-parameter computation | 2.3 |


## 1 · Setup

If you are using Colab, the cell below clones the starter repository from GitHub. If you are running this notebook locally from inside the repository, it skips that step.


In [ ]:
import os, sys

if not os.path.exists('cs687'):
    # Clone the fixed test release used for the student-workflow rehearsal.
    !git -c advice.detachedHead=false clone --branch v0.9-test --depth 1 https://github.com/mk-er/cs687-hw01-p1c1-test.git
    %cd cs687-hw01-p1c1-test

!python -m pip install -q -r requirements.txt
sys.path.insert(0, os.getcwd())
print('ready')


## 2 · Task 1 — the byte-pair-encoding trainer

Before implementing the training loop, open `cs687/tokenizer.py` and read `BPETokenizer.__init__`, `_pretokenize`, and `_apply_merge`. These define the two dictionaries you will update, the structure of `chunks`, and the helper used to apply each learned merge. You do not need to modify those methods.

The cell below imports the supplied `BPETokenizer` class and asks you to implement its missing `train` method.

The algorithm:

1. Pre-tokenize the text into chunks. This is done for you; `chunks` is a list of lists of byte values.
2. Repeat `num_merges` times:
   1. Count how often each pair of adjacent symbols occurs, across all chunks. Pairs never span two chunks.
   2. Stop early if there are no pairs left.
   3. Find the most frequent pair.
   4. Give it a new identifier, starting at 256, so that on step number `step` the new identifier is `256 + step`.
   5. Record the rule in `self.merges`, and record the byte string of the new symbol in `self.vocab`. The byte string of a merged symbol is the concatenation of the byte strings of its two parts.
   6. Rewrite every chunk, replacing each occurrence of the pair with the new identifier.

Two hints. `counts.update(zip(chunk, chunk[1:]))` counts the adjacent pairs of one chunk. `max(counts, key=counts.get)` returns the most frequent pair.

**Write your loop in the cell below, replacing the `raise NotImplementedError` line.**


In [ ]:
from collections import Counter
from cs687.tokenizer import BPETokenizer


def train(self, text: str, num_merges: int) -> None:
    """Learn `num_merges` BPE merge rules from `text`.

    Follow the algorithm described above. Update `self.merges` and
    `self.vocab`, and use `_apply_merge` to rewrite every chunk.
    """
    chunks = self._pretokenize(text)

    # ================= YOUR CODE STARTS HERE =================
    raise NotImplementedError('Task 1: implement the training loop')
    # ================= YOUR CODE ENDS HERE ===================


# Attach your version to the class so the rest of the notebook uses it.
BPETokenizer.train = train
print('train() attached')


In [ ]:
# Check Task 1 before continuing. These tests use the implementation
# attached to BPETokenizer by the answer cell above.
import pytest

task1_result = pytest.main(['-q', 'tests/test_tokenizer.py'])
if task1_result != pytest.ExitCode.OK:
    raise AssertionError('Task 1 has not passed all public tests. Review the failures above.')

print('Task 1 passed all public tests.')


### Try it

A small corpus, a few merges, and a look at what was learned.


In [ ]:
tok = BPETokenizer()
tok.train('the model predicts the next token in the sequence ' * 30, 20)

print(f'learned {len(tok.merges)} merge rules:')
for n, new_id in enumerate(sorted(tok.merges.values()), start=1):
    print(f'  {n:2d}. {tok.vocab[new_id]!r}')


## 3 · Checking against the hand trace

This is the corpus from the lecture: `hug` ten times, `pug` five, `pun` twelve, `bun` four, `hugs` five.

If your loop is correct, the first three merges are `ug`, then `un`, then `hug` — exactly what was worked on the board.

The words are separated by newlines rather than spaces, so that each word is seen in isolation. With spaces, the algorithm would also be free to learn a pair such as (space, p), which is correct behaviour but is not what the trace on the board shows.


In [ ]:
corpus = '\n'.join(['hug'] * 10 + ['pug'] * 5 + ['pun'] * 12 + ['bun'] * 4 + ['hugs'] * 5)

trace = BPETokenizer()
trace.train(corpus, 3)
learned = [trace.vocab[i] for i in sorted(trace.merges.values())]

print('learned:', learned)
print('expected:', [b'ug', b'un', b'hug'])
assert learned == [b'ug', b'un', b'hug'], 'does not match the hand trace yet'
print('\nMatches the hand trace.')


### The open-vocabulary property

The word `bug` never appears in the corpus. It can still be written down, because the base symbols are the raw byte values. Nothing is ever unrepresentable.


In [ ]:
for word in ['hugs', 'bug', 'evlerinizden', 'tokens \U0001f9e9']:
    ids = trace.encode(word)
    pieces = [trace.decode([i]) for i in ids]
    print(f'{word!r:22} -> {len(ids):2d} tokens: {pieces}')
    assert trace.decode(ids) == word
print('\nEvery round trip is exact. The code is lossless.')


## 4 · Comparing with GPT-2's tokenizer

Your tokenizer has learned a few hundred merges. GPT-2's has learned about fifty thousand, from billions of words. Find three strings where GPT-2 needs one token and yours needs several, and think about why.


In [ ]:
# GPT-2's tokenizer is downloaded the first time it is used, so this cell
# needs an internet connection. If it is not available, the notebook carries
# on without the GPT-2 comparison columns.
try:
    import tiktoken
    gpt2 = tiktoken.get_encoding('gpt2')
    HAVE_GPT2 = True
except Exception as error:
    gpt2, HAVE_GPT2 = None, False
    print('GPT-2 tokenizer unavailable:', type(error).__name__)
    print('The homework still works. The comparison columns will be skipped.')

yours = BPETokenizer()
yours.train(open('data/parallel_en_tr.tsv', encoding='utf-8').read(), 500)

examples = ['tokenization', 'strawberry', 'the', ' the', 'university', 'evlerinizden']
if HAVE_GPT2:
    print(f"{'string':18} {'GPT-2':>7} {'yours':>7}")
    print('-' * 34)
    for s in examples:
        print(f'{s!r:18} {len(gpt2.encode(s)):7d} {len(yours.encode(s)):7d}')
else:
    print(f"{'string':18} {'yours':>7}")
    print('-' * 26)
    for s in examples:
        print(f'{s!r:18} {len(yours.encode(s)):7d}')


### The leading-space artifact

Section 2.3 of the notes: `' the'` and `'the'` are different tokens, with independently learned representations. Pre-tokenization is the reason.


In [ ]:
for s in ['the', ' the']:
    gpt2_ids = gpt2.encode(s) if HAVE_GPT2 else 'unavailable'
    print(f'{s!r:8} -> GPT-2 {gpt2_ids}   yours {yours.encode(s)}')

print('\nThe two are different tokens, with independently learned representations.')


### Why a model miscounts the letters in *strawberry*

Look at the pieces the model actually receives. It is being asked about characters inside symbols it has no way of taking apart.


In [ ]:
target = gpt2 if HAVE_GPT2 else yours
name = 'GPT-2' if HAVE_GPT2 else 'your tokenizer'
ids = target.encode('strawberry')
print(f'pieces {name} gives the model:', [target.decode([i]) for i in ids])
print('the characters were discarded by the interface before the model saw anything')


## 5 · The fertility experiment

The same content, written in two languages, measured with the same tokenizers. Fertility is the average number of tokens per word.

**Copy the table this produces into your report.**


In [ ]:
def load_parallel(path='data/parallel_en_tr.tsv'):
    english, turkish = [], []
    for line in open(path, encoding='utf-8').read().splitlines():
        if not line.strip() or line.startswith('>'):
            continue
        parts = line.split('\t')
        if len(parts) == 2:
            english.append(parts[0].strip())
            turkish.append(parts[1].strip())
    return english, turkish


english, turkish = load_parallel()
print(f'{len(english)} aligned sentence pairs')
print('example:', english[2])
print('        ', turkish[2])


In [ ]:
def fertility(tok, sentences):
    text = ' '.join(sentences)
    return len(tok.encode(text)) / len(text.split())


en_only = BPETokenizer()
en_only.train(' '.join(english), 500)

both = BPETokenizer()
both.train(' '.join(english) + ' ' + ' '.join(turkish), 500)

rows = [('yours, English only', en_only), ('yours, both languages', both)]
if HAVE_GPT2:
    rows.append(('GPT-2', gpt2))

print(f"{'tokenizer':24} {'EN':>7} {'TR':>7} {'ratio':>7}")
print('-' * 48)
for name, tok in rows:
    en = fertility(tok, english)
    tr = fertility(tok, turkish)
    print(f'{name:24} {en:7.2f} {tr:7.2f} {tr/en:7.2f}')


**Write two sentences in your report.** What do these numbers imply about the effective context length, and about the cost per query, for a Turkish-speaking user of an English-centric model?

Notice what changed between the first two rows. The algorithm is identical. Only the training data differed. The imbalance is inherited, not designed.


## 6 · Task 2 — training pairs by sliding window

Autoregressive training needs pairs where the target is the input shifted forward by one position:

```
input  = ids[s     : s + T]        that is, (y_1, ..., y_T)
target = ids[s + 1 : s + T + 1]    that is, (y_2, ..., y_{T+1})
```

One forward pass then produces a loss at every position at once.

Walk a window across `ids` in steps of `stride`, appending one tensor to each list per window. Stop at `len(ids) - context_len`, because the final window needs one extra token for its target.

**Write your loop in the cell below.**


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


class NextTokenDataset(Dataset):
    def __init__(self, ids, context_len, stride):
        self.inputs, self.targets = [], []

        # ================= YOUR CODE STARTS HERE =================
        raise NotImplementedError('Task 2: fill self.inputs and self.targets')
        # ================= YOUR CODE ENDS HERE ===================

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, i):
        return self.inputs[i], self.targets[i]


print('class defined')


In [ ]:
# Make the notebook's Task 2 class visible to its public tests.
import cs687 as cs687_package
import cs687.data as data_module
cs687_package.NextTokenDataset = NextTokenDataset
data_module.NextTokenDataset = NextTokenDataset

import pytest

task2_result = pytest.main(['-q', 'tests/test_data.py'])
if task2_result != pytest.ExitCode.OK:
    raise AssertionError('Task 2 has not passed all public tests. Review the failures above.')

print('Task 2 passed all public tests.')


### Look at one pair

The offset of exactly one position is the whole idea. Confirm it with your own eyes.


In [ ]:
ids = yours.encode(' '.join(english[:12]))
ds = NextTokenDataset(ids, context_len=10, stride=5)

x, y = ds[0]
print('input :', [yours.decode([i]) for i in x.tolist()])
print('target:', [yours.decode([i]) for i in y.tolist()])
print()
print('input ids :', x.tolist())
print('target ids:', y.tolist())
print('\nThe target is the input shifted left by one position.')


### Stride and overlap

A smaller stride produces more windows from the same text, at the cost of correlation between them.


In [ ]:
for stride in [10, 5, 2, 1]:
    n = len(NextTokenDataset(ids, context_len=10, stride=stride))
    print(f'stride {stride:2d} -> {n:4d} training pairs')


## 7 · Embeddings, and the shape check

The final shape check: integers of shape `(B, T)` go in, and floating-point vectors of shape `(B, T, d_model)` come out.


In [ ]:
from cs687.embedding import InputEmbedding

loader = DataLoader(NextTokenDataset(ids, 10, 5), batch_size=4, shuffle=False, drop_last=True)
batch_x, batch_y = next(iter(loader))

emb = InputEmbedding(vocab_size=yours.vocab_size, d_model=64, max_len=128)
out = emb(batch_x)

print('token ids  :', tuple(batch_x.shape))
print('embeddings :', tuple(out.shape))
assert out.shape == (4, 10, 64)
print('\nShape check passed.')


### The lookup really is a linear map

Section 3.1: multiplying the embedding matrix by a one-hot vector selects one column. `nn.Embedding` computes the same thing with an index instead of a multiplication.


In [ ]:
emb.eval()
with torch.no_grad():
    direct = emb.tok(torch.tensor([3]))
    one_hot = torch.zeros(1, yours.vocab_size)
    one_hot[0, 3] = 1.0
    via_matmul = one_hot @ emb.tok.weight

print('largest difference:', float((direct - via_matmul).abs().max()))
print('identical to within floating-point error')


### Position information really is added

The same token at two different positions must not have the same representation, because self-attention treats its inputs as an unordered set.


In [ ]:
with torch.no_grad():
    same_token_twice = emb(torch.tensor([[5, 5]]))

difference = float((same_token_twice[0, 0] - same_token_twice[0, 1]).abs().max())
print('difference between the two positions:', round(difference, 4))
print('non-zero, because the position embedding differs')


## 8 · Units: nats, perplexity, and bits per byte

The worked example from Section 1.3, reproduced in code.


In [ ]:
from cs687.units import perplexity, bits_per_token, bits_per_byte, compression_ratio

loss = 3.2
tokens_per_word, bytes_per_word = 1.3, 5.9
tpb = tokens_per_word / bytes_per_word

print(f'loss            {loss:.2f} nats per token')
print(f'perplexity      {perplexity(loss):.1f}')
print(f'bits per token  {bits_per_token(loss):.2f}')
print(f'tokens per byte {tpb:.3f}')
print(f'bits per byte   {bits_per_byte(loss, tpb):.2f}')
print(f'compresses to   {100 * compression_ratio(loss, tpb):.0f}% of the original size')


### Now with your own tokenizer's numbers

This is exactly the computation asked for in Question D of your report. Substitute the loss your Project 1 model reaches once you have trained it.


In [ ]:
your_loss = 4.1   # replace with your own measured loss
text = ' '.join(english)
your_tpb = len(yours.encode(text)) / len(text.encode('utf-8'))

print(f'your tokens per byte : {your_tpb:.3f}')
print(f'your bits per token  : {bits_per_token(your_loss):.2f}')
print(f'your bits per byte   : {bits_per_byte(your_loss, your_tpb):.2f}')


## 9 · How much of a model is the embedding matrix?

Guess before you run this cell. Most people guess far too low.


In [ ]:
V, d_model, total = 50257, 768, 124_000_000

embedding_params = V * d_model
print(f'{V} x {d_model} = {embedding_params:,} parameters')
print(f'that is {100 * embedding_params / total:.0f}% of a {total:,} parameter model')
print()
untied = total + embedding_params
print(f'if the output projection were not shared: {100 * 2 * embedding_params / untied:.0f}% of the enlarged model')


## 10 · Run the public tests

The two focused test cells above provided immediate feedback after each answer. This final cell runs all 31 public tests together against the implementations currently loaded in the notebook. Always run it before downloading and submitting your notebook. It does not require you to copy anything into the repository's `.py` files. Staff grading will use a separate, staff-controlled test suite.


In [ ]:
# Make the notebook's Task 2 class visible to the public tests.
# Task 1 was attached to BPETokenizer in its answer cell above.
import cs687 as cs687_package
import cs687.data as data_module
cs687_package.NextTokenDataset = NextTokenDataset
data_module.NextTokenDataset = NextTokenDataset

import pytest

pytest_result = pytest.main(['-q', 'tests'])
if pytest_result != pytest.ExitCode.OK:
    raise AssertionError('One or more public tests failed. Review the failures above.')

print('All 31 public tests passed.')


## 11 · Finishing the homework

Your implementations in the two tagged answer cells are the code you submit. You do **not** need to copy them into the repository's `.py` files.

Before submitting, restart the Colab runtime and run the notebook from top to bottom. Confirm that the public-test cell reports that all 31 tests passed. Then download the completed `.ipynb` file from Colab and complete the separate `REPORT.md` template. The exact Moodle filenames and upload procedure will be announced before release.

The comprehension questions in the report are the part that matters most. You may write the code with the help of a coding agent, but the quiz and the examination ask the same questions without assistance, so make sure you can explain whatever you submit. Saved notebook output is not used as proof of correctness: the submitted notebook will be executed again from a clean environment.
